# Chinee apple: supervised probe on frozen DINOv3Stage 2 of the *model first, labels later* plan. A logistic-regression head isfitted on **frozen** DINOv3 patch tokens using the CVAT + SAM3 polygonannotations, then scored at three levels: patch, mask and instance (COCO mAP).The backbone is never fine-tuned. Only the head is trained, so this runs on afree Colab T4 in a few minutes.**What you need**1. The annotated tiles, and the CVAT `batch_*.zip` exports in the same folder.2. Optionally DINOv3 weights. They are gated, so the notebook falls back to   ungated DINOv2 and everything downstream is identical.

## 1. Environment

In [ ]:
!nvidia-smi -L || echo "no GPU, this still runs on CPU but slower"!pip -q install scikit-learn joblib pillow scipyimport torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 2. CodeClone the repo so `vit_features.py`, `train_probe.py` and `probe_eval.py` areimportable. Replace the URL if you work from a fork.

In [ ]:
import os, sysREPO = "https://github.com/nutboltu/chinee-apple-cv-benchmark.git"if not os.path.exists("chinee-apple-cv-benchmark"):    !git clone -q $REPOsys.path.insert(0, "chinee-apple-cv-benchmark/dinov3-vit")import vit_features as vf, train_probe as tp, probe_eval as peprint("ok")

## 3. DataThe imagery is not in git. Point `DATA` at a folder holding the tile jpgs(in `batch_01/`.. subfolders or flat) **and** the `batch_*.zip` CVAT exports.Easiest is Google Drive.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')from pathlib import PathDATA = Path('/content/drive/MyDrive/chinee-apple/processed')   # <- edit meCRIB = DATA / 'tile_markers.csv'        # optional, masks cardboard outassert DATA.exists(), f"not found: {DATA}"print("zips :", len(list(DATA.glob('batch_*.zip'))))print("tiles:", len(list(DATA.rglob('*.jpg'))))

## 4. BackboneDINOv3 is gated. Accept Meta's licence, clone their repo and downloadViT-B/16, then set `FAMILY='dinov3'` with `REPO`/`WEIGHTS`. Until thenDINOv2 ViT-B/14 loads straight from torch.hub and the rest is unchanged.

In [ ]:
FAMILY   = 'dinov2'      # 'dinov3' once you have the weightsREPO_DIR = None          # e.g. '/content/dinov3'WEIGHTS  = None          # e.g. '/content/dinov3_vitb16.pth'IMG_SIZE = 448           # 448 is a good speed/detail balance; 768 is sharper and ~3x slowermodel, meta = vf.load_backbone(family=FAMILY, repo=REPO_DIR, weights=WEIGHTS)print(meta)

## 5. Features and per-patch labelsEach tile goes through the frozen backbone once. Polygons are rasterised,downsampled onto the token grid, and each patch is labelled by how much of itthe annotation covers.Patches straddling a polygon edge are **dropped**, not forced into a class.Cardboard markers are punched out of the masks, because a head that learned"white cardboard" would score well here and fail on unmarked imagery.

In [ ]:
import numpy as npfrom PIL import ImagePOS_T, NEG_T = 0.50, 0.10polys = tp.read_cvat_zips([str(DATA / 'batch_*.zip')])tiles = tp.index_tiles(DATA)crib  = tp.read_marker_boxes(CRIB) if CRIB.exists() else {}names = [n for n in sorted(polys) if n in tiles]print(f"annotated tiles with imagery: {len(names)}")recs = []for i, n in enumerate(names, 1):    f, (gh, gw) = vf.patch_features(model, str(tiles[n]), meta, img_size=IMG_SIZE)    f = np.asarray(f, dtype=np.float32)    f /= (np.linalg.norm(f, axis=1, keepdims=True) + 1e-8)    with Image.open(tiles[n]) as im: size = im.size    side = gh * meta['patch']    cov  = tp.patch_coverage(tp.tile_mask(size, polys[n], crib.get(n)), side, meta['patch'])    inst = []    for pts in polys[n]:        g = tp.patch_coverage(tp.tile_mask(size, [pts], crib.get(n)), side, meta['patch']) >= 0.5        if g.any(): inst.append(g)    recs.append(dict(name=n, feat=f, cov=cov, grid=(gh, gw), inst=inst, src=tp.source_of(n)))    if i % 25 == 0: print(f"  {i}/{len(names)}")print(f"done. grid {recs[0]['grid']}, dim {recs[0]['feat'].shape[1]}, frames {len({r['src'] for r in recs})}")

## 6. Split by source frame**This matters more than anything else in the notebook.** Tiles are cut 8x6from one 8192x5460 frame, and the bottom tile row is shifted flush to thebottom edge, so it overlaps the row above by 684 px. Neighbouring tiles fromone frame share content, and in that band share literal pixels.Splitting randomly over tiles or patches puts near-duplicates on both sides andinflates every metric below. Group by source frame.

In [ ]:
from sklearn.model_selection import GroupShuffleSplitfrom sklearn.linear_model import LogisticRegressionG = np.array([r['src'] for r in recs])tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)              .split(np.zeros(len(recs)), groups=G))def pack(idx):    X, Y = [], []    for j in idx:        r = recs[j]; y = tp.label_patches(r['cov'], POS_T, NEG_T); k = y >= 0        X.append(r['feat'][k]); Y.append(y[k])    return np.concatenate(X), np.concatenate(Y)Xtr, Ytr = pack(tr)clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0).fit(Xtr, Ytr)print(f"train {len(set(G[tr]))} frames / {len(tr)} tiles / {len(Ytr)} patches ({100*Ytr.mean():.1f}% positive)")print(f"test  {len(set(G[te]))} frames / {len(te)} tiles")

## 7. Metrics- **patch** AP (= mAP, one class), ROC AUC, precision, recall, F1- **mask** mean IoU and Dice per tile- **instance** COCO-style mAP@[.50:.95], AP@0.50, AP@0.75 over connected componentsAll scored on the token grid, which is the probe's real resolution. Upsamplingpredictions to pixels before scoring would flatter the boundary.

In [ ]:
ys, ps, tl = [], [], []for j in te:    r = recs[j]; gh, gw = r['grid']    prob = clf.predict_proba(r['feat'])[:, 1]    y = tp.label_patches(r['cov'], POS_T, NEG_T); k = y >= 0    ys.append(y[k]); ps.append(prob[k])    tl.append({'prob': prob.reshape(gh, gw), 'gt': r['cov'] >= 0.5, 'gt_instances': r['inst']})Y = np.concatenate(ys); P = np.concatenate(ps)res = pe.report(Y, P, tl)

### Precision / recall curve

In [ ]:
import matplotlib.pyplot as pltfrom sklearn.metrics import precision_recall_curvepr, rc, _ = precision_recall_curve(Y, P)plt.figure(figsize=(5,4))plt.plot(rc, pr, lw=2)plt.axhline(Y.mean(), ls='--', c='gray', lw=1, label=f'chance {Y.mean():.2f}')plt.xlabel('recall'); plt.ylabel('precision')plt.title(f"patch PR, AP={res['patch']['ap']:.3f}")plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

### Instance AP against post-processingA per-patch head has no spatial prior and fragments single canopies, so closing the mask before labelling components matters a lot. Check it rather than trusting the default.

In [ ]:
print(f"{'smooth':>7}{'min_size':>10}{'mAP@.5:.95':>12}{'AP@.50':>9}{'preds':>8}")for sm in (0,1,2,3):    for ms in (4,8,16):        im = pe.instance_map(tl, res['patch']['threshold'], min_size=ms, smooth=sm)        print(f"{sm:>7}{ms:>10}{im['mAP_50_95']:>12.3f}{im['AP_50']:>9.3f}{im['n_pred_instances']:>8}")print(f"\nground truth instances: {sum(len(t['gt_instances']) for t in tl)}")

### Qualitative checkNumbers hide failure modes. Look at the heatmaps.

In [ ]:
import matplotlib.pyplot as pltfrom PIL import Imageshow = te[:4]fig, ax = plt.subplots(2, len(show), figsize=(4*len(show), 8))for c, j in enumerate(show):    r = recs[j]; gh, gw = r['grid']    prob = clf.predict_proba(r['feat'])[:, 1].reshape(gh, gw)    ax[0, c].imshow(Image.open(tiles[r['name']]).resize((320, 320)))    ax[0, c].set_title(r['name'][-18:], fontsize=8); ax[0, c].axis('off')    ax[1, c].imshow(prob, cmap='magma', vmin=0, vmax=1)    ax[1, c].contour(r['cov'] >= 0.5, levels=[.5], colors='lime', linewidths=1.2)    ax[1, c].set_title('p(weed), green = ground truth', fontsize=8); ax[1, c].axis('off')plt.tight_layout(); plt.show()

## 8. Save

In [ ]:
import joblib, jsonOUT = Path('/content/drive/MyDrive/chinee-apple/probe_out'); OUT.mkdir(parents=True, exist_ok=True)joblib.dump({'clf': clf, 'family': FAMILY, 'img_size': IMG_SIZE,             'patch': meta['patch'], 'l2_normalized': True}, OUT / 'probe.joblib')clean = {k: {kk: (float(vv) if isinstance(vv, (int, float, np.floating)) else str(vv))             for kk, vv in v.items() if kk != 'per_iou'} for k, v in res.items()}(OUT / 'metrics.json').write_text(json.dumps(clean, indent=2))print('saved to', OUT)

## Notes and limits**Empty tiles are excluded on purpose.** 440 of the 620 tiles came back with noannotation. Under the blue-only protocol an empty tile means nobody annotatedit, not that it holds no chinee apple, so using them as background would trainthe head to suppress real plants. Every negative here comes from inside anannotated tile.**Instance mAP is far below patch AP**, and that is the honest picture: this isa decent per-patch classifier and a weak instance detector. The head has nospatial prior, so canopies fragment. If instance quality is the goal, the nextstep is a light segmentation decoder on the frozen tokens, not a betterthreshold.**The baseline to beat.** DINOv2 ViT-B/14 at 448px, 180 tiles, 56 frames,grouped split: patch AP ~0.77, mask IoU ~0.50, instance mAP@[.50:.95] ~0.15.Real DINOv3 weights should improve on this.